# Watermarks and Late-Arriving Data: Bounding Streaming State Memory

**Module**: 09_streaming  
**Notebook**: 04_watermarks_and_late_data.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How watermarks bound state store memory during streaming aggregations and drop late-arriving events.
- **Why does this topic exist?**: Without watermarks, streaming aggregations maintain historical state indefinitely, eventually crashing with OOM.
- **What problem does it solve?**: State store growing unbounded on high-velocity event streams.
- **Where is it used in real-world Data Engineering?**: Sessionizing user clickstreams and windowed IoT metrics.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Streaming aggregations, event time, output modes |
| **Required depth** | Working knowledge: can use watermarks to bound state and handle late data |
| **Data Engineer** | Choose a watermark delay and explain late-data behavior. |
| **Senior Data Engineer** | Diagnose unbounded state and incorrect window results. |
| **Spark specialist** | Reason about watermark propagation, state cleanup, and append-mode emission. |
| **Next topic** | `09_streaming/05_checkpoints_and_fault_tolerance.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Event-time processing must decide how long to wait for late data. A watermark sets a bound: Spark tracks the
maximum event time seen and drops state older than `max event time - delay`. A larger delay tolerates more
lateness but keeps more state; a smaller delay reduces state but drops more late events. There is no universal
correct value.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Event time** | When the event occurred | Embedded in the record, not the arrival time |
| **Watermark** | Lateness bound | `max event time - delay` |
| **Late data** | Events below the watermark | Dropped from stateful operations |
| **State cleanup** | Remove old windows | Watermark enables bounded state |
| **Window** | Time bucket | Tumbling, sliding, or session |
| **Append emission** | Emit after watermark | A window result is finalized once the watermark passes |

### How it works, step by step

1. The query reads events with an event-time column.
2. `withWatermark` defines the allowed lateness.
3. Spark tracks the maximum event time seen.
4. State for windows older than the watermark is eligible for cleanup.
5. Late events below the watermark are ignored by stateful operators.
6. In append mode, a window result is emitted once the watermark passes its end.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Watermark Mechanics
`Watermark = MaxEventTimeSeen - AllowedDelay`
- Events with `event_time >= Watermark` are accepted into the aggregation window.
- Events with `event_time < Watermark` are discarded as too late.
- State for windows older than the watermark is cleared from memory.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Watermarks_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Define streaming query with 10-minute watermark and 5-minute tumbling window
rate = spark.readStream.format("rate").option("rowsPerSecond", 2).load()

windowed = (
    rate
    .withWatermark("timestamp", "10 seconds")
    .groupBy(F.window("timestamp", "5 seconds"))
    .count()
)

print("Streaming plan with Watermark:")
windowed.explain()

---

## Experiment: Running Watermarked Aggregation to Memory Sink

- **1. What are we testing?**: Run windowed aggregation with watermark for 6 seconds.
- **2. What do we expect?**: Computes window counts and prunes expired windows.

In [ ]:
q = windowed.writeStream.format("memory").queryName("window_sink").outputMode("update").start()
time.sleep(6)
spark.sql("SELECT window.start, window.end, count FROM window_sink ORDER BY window.start DESC LIMIT 5").show(truncate=False)
q.stop()

**3. What happened?**: Output windows emitted incrementally.  
**4. Why did it happen?**: Update mode emits changed window counts as rows arrive.  
**5. Practical Engineering Lesson**: Always define withWatermark() on all stateful streaming aggregations.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Late sensor readings

- **Industry**: Industrial IoT
- **Scenario**: Sensors buffer readings and send them minutes late after connectivity outages.
- **Problem**: A short watermark drops legitimate readings; a long watermark keeps excessive state.
- **Why this concept is relevant**: The delay should reflect the observed lateness distribution, not a guess.
- **Implementation idea**: Measure event-time lateness percentiles and set the watermark to cover the agreed SLA.
- **Expected result**: Bounded state with an acceptable late-data loss rate.
- **Alternative**: Keep all state and reconcile in batch later.
- **Trade-offs**: Batch reconciliation improves completeness but adds latency and duplicate-handling complexity.

### Use Case 2: Sessionization with mobile events

- **Industry**: Mobile apps
- **Scenario**: Events arrive out of order because devices retry after reconnect.
- **Problem**: Without a watermark, session state grows forever and the job eventually fails.
- **Why this concept is relevant**: Watermarking bounds session state and defines which late events are ignored.
- **Implementation idea**: Use event time, a session window, and a watermark based on the retry window.
- **Expected result**: Stable state size and predictable output.
- **Alternative**: A longer state store with periodic compaction.
- **Trade-offs**: A larger state store handles more lateness but increases memory and checkpoint cost.

---

## Business Value

- Bounded state and stable long-running jobs.
- A measurable trade-off between completeness and cost.
- Correct event-time semantics for out-of-order data.
- Clear SLAs for late-data handling.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Set the watermark from measured lateness, not from a copied default.
- Monitor late-data drop rates.
- Use event time, not ingestion time, for business windows.
- Remember that watermarks only affect stateful operations.
- Test the behavior at the watermark boundary.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Measure late-data loss

**Steps**

1. Create events with a controlled fraction arriving after the watermark.
2. Run the aggregation with a short and a long delay.
3. Compare state size, output completeness, and dropped-event counts.

**What to observe**: A longer delay keeps more state and accepts more late events; a shorter delay does the opposite.

**Healthy result**: The delay matches the business lateness tolerance.

**Problematic result**: State grows without bound or legitimate events are dropped.

**Engineering takeaway**: The watermark is a business decision with a memory cost.

---

## Trade-offs and Alternatives

### Advantages

- Bounds state
- Handles out-of-order data
- Enables append-mode emission

### Limitations

- Drops late data
- Delay is workload-specific
- Does not guarantee completeness

### When to use it

- Windowed aggregations
- Sessionization
- IoT telemetry

### When not to use it

- Arbitrary delays
- Ignoring late-data metrics
- Expecting exactly-once completeness without reconciliation

### Alternatives inside Spark

- withWatermark
- Window operations
- State store

### Alternatives outside Spark

- Batch reconciliation
- Streaming databases
- Lambda architecture

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Watermark Syntax
**Problem Statement**: Write withWatermark on timestamp column for 1 hour.  
**Input / Context**: code

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Tumbling vs Sliding Window
**Problem Statement**: Explain window('timestamp','10 minutes','5 minutes').  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): State Eviction Inspection
**Problem Statement**: Check state memory in streaming query progress.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### State never stops growing

**Scenario**: A streaming aggregation's state store grows until the job fails, even though a watermark is set.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check that the watermark is on the event-time column used by the stateful operation.
2. Verify that the query actually uses event time and not processing time.
3. Look for a non-windowed aggregation that retains all keys.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Watermark Syntax):
```python
print('df.withWatermark("timestamp","1 hour")')
```

#### Solution 2 (Tumbling vs Sliding Window):
```python
print('10-minute window sliding every 5 minutes (sliding window).')
```

#### Solution 3 (State Eviction Inspection):
```python
print('q.lastProgress["stateOperators"]')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Watermark Purpose**: Purges old state from RAM to prevent OOM.
- **Late Data**: Data older than watermark is dropped.

### Essential Technical Terms
- `Watermark`: MaxEventTime - AllowedDelay threshold.
- `State Store`: In-memory or RocksDB state repository.

### Common Engineering Mistakes
- **Omitting Watermark on GroupBy**: State store grows indefinitely until executor crashes.

### Interview Questions on This Topic
- **Q: What problem does watermarking solve in Structured Streaming?**
  - *A*: It bounds state memory by specifying how long to wait for late data before purging historical aggregate windows from the state store.


### More Interview Questions

- **Q: How is the watermark computed?**
  - *A*: Spark tracks the maximum event time seen and subtracts the allowed delay; state older than that boundary becomes eligible for cleanup.
- **Q: Why does a watermark drop late data?**
  - *A*: Stateful operations cannot keep every window forever; events below the watermark are considered too late and are ignored to bound state.

### What to Study Next
Open 05_checkpoints_and_fault_tolerance.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')